# Arch 2 - Encoder-decoder language model (from scratch, 2x T4, ~1.5 h budget)

A compact T5-style model written in plain PyTorch ops (no model or tokenizer libraries). **109.5M parameters.**
It is shaped like the Phase 1 task itself: the **encoder reads the 512-word prompt** with bidirectional attention and the
**decoder writes the 128-word continuation**, attending to the prompt through cross-attention.

| Piece | Choice |
|---|---|
| Layout | 6 encoder + 6 decoder layers, d_model 768, 12 query / 4 KV heads (grouped-query attention) |
| Training example | a 1344-token window: first 1024 tokens go to the encoder, the next 320 are the decoder's target (cross-entropy on those) |
| Efficiency | RMSNorm, RoPE, SwiGLU, fused SDPA attention, shared embedding table, fp16 + `torch.compile`, fused AdamW, DDP over both T4s |
| Generation | encoder runs once per prompt; decoder uses a KV cache and precomputed cross-attention K/V; padded prompts are masked |
| **Time budget** | the LR schedule follows `max(step progress, wall-clock progress)`: the cosine decay lands on its floor exactly when `TRAIN_TIME_BUDGET_MIN` (80 min, incl. compile) runs out |
| Safety | checkpoints pushed to Hugging Face every 200 steps, best-validation weights saved separately, exact resume |

Tokenizer and token files are reused from your arch 3 repo (`HF_DATA_REPO` in `config.py`), so every model sees identical data.

**If Kaggle crashes: press Run All again** - it restores everything from the Hub and continues; time already spent counts against the budget.

Settings: GPU **T4 x2**, Internet **on**, secret `HF` holding a write token.

## 1. Setup and Hugging Face login

In [ ]:
!pip install -q ijson
import os, sys, json, math, time, gc, zipfile
import torch
from huggingface_hub import login, HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN          # child processes (torchrun) read it from here
login(token=HF_TOKEN)

WORK_DIR = "/kaggle/working/arch2"
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, WORK_DIR)
print("Logged in to Hugging Face.")

## 2. GPU check

In [ ]:
n_gpu = torch.cuda.device_count()
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB, compute capability {p.major}.{p.minor}")
if n_gpu < 2:
    print("Only one GPU visible - training still works, it just runs gradient accumulation instead of DDP.")

## 3. Configuration
Every knob lives in `config.py`. Edit it here, then re-run this cell.

In [ ]:
%%writefile /kaggle/working/arch2/config.py
"""
Single place for every knob in the arch-2 (encoder-decoder) pipeline.
All scripts (tokenizer, data prep, model, trainer) import this file.
"""
import os
import json

# ---------------------------------------------------------------- hub / data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO = "D-240357/legal-arch2-encdec"    # data (if built here), checkpoints and final model are mirrored here
HF_DATA_REPO = "D-240357/legal-arch3-decoder"    # arch 3 already built the tokenizer + token files: reuse them (same data, fair comparison)
HF_PRIVATE = True

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Relative share of the training text taken from each file above (same order). The corpus is
# interleaved by these weights, so 600M tokens cover all four sources instead of just the first
# files in the list. Weights are by characters; a source that runs out is simply skipped.
SOURCE_WEIGHTS = [1.0, 1.0, 1.0, 1.0]

# ------------------------------------------------------------------- folders
WORK_DIR = "/kaggle/working/arch2"
DATA_DIR = os.path.join(WORK_DIR, "data")
CKPT_DIR = os.path.join(WORK_DIR, "ckpt")
CODE_DIR = WORK_DIR

TOKENIZER_PATH = os.path.join(DATA_DIR, "tokenizer.json")
TRAIN_BIN = os.path.join(DATA_DIR, "train.bin")
VAL_BIN = os.path.join(DATA_DIR, "val.bin")
META_PATH = os.path.join(DATA_DIR, "meta.json")
STATE_PATH = os.path.join(CKPT_DIR, "training_state.pt")
FINAL_PATH = os.path.join(CKPT_DIR, "model_final.pt")
LOG_PATH = os.path.join(CKPT_DIR, "train_log.jsonl")
BEST_PATH = os.path.join(CKPT_DIR, "model_best.pt")     # weights with the lowest validation loss so far

# ----------------------------------------------------------------- tokenizer
VOCAB_SIZE = 32000            # 256 bytes + merges + 2 specials (32000 = 500 * 64, tensor-core friendly)
TOK_SAMPLE_MB_PER_FILE = 40   # text used to learn the merges, taken from each train file
TOK_SAMPLE_STRIDE = 5         # take every 5th doc so the sample spans the whole file

# ---------------------------------------------------------------------- data
TRAIN_TOKENS = 150_000_000    # upper bound on the text consumed; the time budget below normally ends training first
VAL_TOKENS = 3_000_000
VAL_EVERY_NTH_DOC = 100       # every 100th document is held out for validation
MIN_DOC_CHARS = 100
PREP_WORKERS = max(1, (os.cpu_count() or 2))

# --------------------------------------------------------------------- model
# Encoder-decoder: the encoder reads the prompt, the decoder writes the continuation.
D_MODEL = 768
N_HEAD = 12
N_KV_HEAD = 4                 # grouped-query attention (3 query heads per KV head)
N_ENC_LAYER = 6
N_DEC_LAYER = 6
D_FF = 2048                   # SwiGLU hidden size
ENC_LEN = 1024                # prompt tokens (a 512-word prompt is ~700-900 BPE tokens)
DEC_LEN = 320                 # continuation tokens per training example (128 words is ~170-200 tokens)
ROPE_BASE = 10000.0
NORM_EPS = 1e-5
EXAMPLE_LEN = ENC_LEN + DEC_LEN   # tokens read from the corpus per training example

# ------------------------------------------------------------------ training
SEED = 42
MICRO_BATCH = 4               # examples per GPU per forward pass
EXAMPLES_PER_STEP = 48        # global batch (48 x 1344 = ~64.5K corpus tokens); grad-accum is derived from the GPU count
PEAK_LR = 6e-4
MIN_LR_RATIO = 0.1
WARMUP_STEPS = 60
WEIGHT_DECAY = 0.1
BETAS = (0.9, 0.95)
MAX_GRAD_NORM = 1.0
Z_LOSS = 1e-4                 # tiny penalty on logsumexp^2, keeps fp16 logits tame

# Hard wall-clock budget. The LR schedule follows max(step progress, time progress), so the cosine always
# reaches its floor exactly when the budget runs out: training ends on time with a fully decayed LR.
TRAIN_TIME_BUDGET_MIN = 80    # trainer wall-clock incl. torch.compile warm-up; leaves ~10 min inside a 1.5 h limit

COMPILE = True
ENFORCE_PARAM_BUDGET = True   # capstone rule: 50M-300M parameters
LOG_EVERY = 20
EVAL_EVERY = 100
EVAL_SEQS = 64                # validation examples per eval (split across GPUs)
CKPT_EVERY = 200              # steps between local save + HF push
SQUASH_EVERY_N_PUSHES = 8     # collapse HF git history now and then so old copies get freed
TIME_LIMIT_HOURS = 10.5       # crash guard only (Kaggle's 12h wall); the budget above is what normally ends the run
NCCL_P2P_OFF = True           # P2P between the two T4s hangs on some Kaggle images; comm cost is tiny here

# --------------------------------------------------------------- generation
TARGET_WORDS = 128
MAX_NEW_TOKENS = 320
GEN_TEMPERATURE = 0.7
GEN_TOP_K = 50
GEN_TOP_P = 0.92
N_LOCAL_EVAL = 48              # held-out validation docs used for the local generation check
GEN_BATCH = 16

# Handy for smoke tests: ARCH3_CFG_OVERRIDE='{"N_ENC_LAYER": 2, ...}' patches any value above.
_override = os.environ.get("ARCH3_CFG_OVERRIDE")
if _override:
    globals().update(json.loads(_override))

EXAMPLE_LEN = ENC_LEN + DEC_LEN   # (re)computed after overrides so smoke tests stay consistent

## 4. Restore anything already saved on Hugging Face
On a fresh run this does nothing. After a crash it brings back the tokenizer, token files and the latest checkpoint.

In [ ]:
import importlib
import config as C
importlib.reload(C)

for d in (C.DATA_DIR, C.CKPT_DIR):
    os.makedirs(d, exist_ok=True)

api = HfApi()
api.create_repo(C.HF_MODEL_REPO, token=HF_TOKEN, private=C.HF_PRIVATE, exist_ok=True)
on_hub = set(api.list_repo_files(C.HF_MODEL_REPO, token=HF_TOKEN))

def restore(remote_name, local_path, local_dir):
    if os.path.exists(local_path):
        print(f"  have locally: {remote_name}")
    elif remote_name in on_hub:
        print(f"  downloading {remote_name} ...")
        hf_hub_download(C.HF_MODEL_REPO, remote_name, local_dir=local_dir, token=HF_TOKEN)
    else:
        return False
    return True

# 'data/meta.json' is uploaded last, so its presence means the whole data upload finished
def hub_files(repo):
    try:
        return set(api.list_repo_files(repo, token=HF_TOKEN))
    except Exception:
        return set()

data_repo = None
for repo in (C.HF_MODEL_REPO, C.HF_DATA_REPO):
    if "data/meta.json" in hub_files(repo):
        data_repo = repo
        break

if data_repo:
    print(f"  using tokenizer + token files from {data_repo}")
    for name in ("tokenizer.json", "val.bin", "train.bin", "meta.json"):
        local = os.path.join(C.DATA_DIR, name)
        if os.path.exists(local):
            print(f"  have locally: data/{name}")
        else:
            print(f"  downloading data/{name} ...")
            hf_hub_download(data_repo, f"data/{name}", local_dir=C.WORK_DIR, token=HF_TOKEN)
else:
    print("  no prepared data on the hub yet - section 6 will build it")

if restore("training_state.pt", C.STATE_PATH, C.CKPT_DIR):
    restore("train_log.jsonl", C.LOG_PATH, C.CKPT_DIR)
    print("Checkpoint found: training will resume from it.")
else:
    print("No checkpoint yet: training will start from scratch.")

## 5. Tokenizer: byte-level BPE from scratch
Pre-token regex -> UTF-8 bytes -> learned merges. Training works on the table of unique words, which keeps pure Python fast enough.

In [ ]:
%%writefile /kaggle/working/arch2/bpe_tokenizer.py
"""
Byte-level BPE tokenizer, written from scratch (stdlib only for the tokenizer logic).

How it works, in short:
  1. A regex chops text into "pre-tokens" (a word with its leading space, a run of digits,
     a run of punctuation, ...). Merges never cross these boundaries.
  2. Every pre-token is turned into raw UTF-8 bytes (ids 0..255), so *any* text can be
     encoded - there is no <unk> and no lowercasing.
  3. Training repeatedly merges the most frequent adjacent id pair into a new id.
     We train on the table of *unique* pre-tokens with their counts, which is what keeps
     it fast in pure Python.
  4. Encoding replays the learned merges (lowest rank first) per pre-token, and caches the
     result, so repeated words cost a single dict lookup.
"""
import re
import json
import heapq
from collections import Counter, defaultdict

# Close to the GPT-2 splitter, but built on plain `re` (no \p{L}), so it needs no extra package.
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+",
    re.UNICODE,
)

EOS_TOKEN = "<|endoftext|>"
PAD_TOKEN = "<|pad|>"


class ByteBPE:
    def __init__(self, merges=None, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = [tuple(m) for m in (merges or [])]      # merge i creates id 256 + i
        self.eos_id = vocab_size - 2
        self.pad_id = vocab_size - 1
        self._cache = {}
        self._rebuild_tables()

    # ------------------------------------------------------------- internals
    def _rebuild_tables(self):
        self.ranks = {pair: (i, 256 + i) for i, pair in enumerate(self.merges)}
        vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            vocab.append(vocab[a] + vocab[b])
        self.vocab = vocab
        self._cache.clear()

    # -------------------------------------------------------------- training
    @classmethod
    def train(cls, texts, vocab_size=32000, log_every=2000):
        """Learn merges from an iterable of strings."""
        n_merges = vocab_size - 256 - 2          # leave room for the two special tokens
        word_counts = Counter()
        for text in texts:
            word_counts.update(PRETOKEN_RE.findall(text))
        print(f"  unique pre-tokens: {len(word_counts):,}")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_count = defaultdict(int)
        where = defaultdict(set)                 # pair -> indices of words that (may) contain it
        for wi, w in enumerate(words):
            f = freqs[wi]
            for pair in zip(w, w[1:]):
                pair_count[pair] += f
                where[pair].add(wi)

        # Max-heap with lazy invalidation: stale entries are fixed up when they surface.
        heap = [(-c, p) for p, c in pair_count.items()]
        heapq.heapify(heap)

        merges = []
        vocab = [bytes([i]) for i in range(256)]
        for m in range(n_merges):
            best = None
            while heap:
                neg, pair = heapq.heappop(heap)
                real = pair_count.get(pair, 0)
                if real <= 0:
                    continue
                if real != -neg:
                    heapq.heappush(heap, (-real, pair))
                    continue
                best = pair
                break
            if best is None:
                print(f"  ran out of pairs after {m} merges")
                break

            new_id = 256 + m
            merges.append(best)
            a, b = best
            vocab.append(vocab[a] + vocab[b])
            touched = {}
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                # find the occurrences in this word (left to right, non-overlapping)
                if len(w) < 2:
                    continue
                i, out, changed = 0, [], False
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                        changed = True
                    else:
                        out.append(w[i])
                        i += 1
                if not changed:
                    continue                      # stale entry in `where`
                for pair in zip(w, w[1:]):        # take the old word's pairs away ...
                    pair_count[pair] -= f
                for pair in zip(out, out[1:]):    # ... and add the new word's pairs
                    pair_count[pair] += f
                    where[pair].add(wi)
                    touched[pair] = pair_count[pair]
                words[wi] = out
            for pair, c in touched.items():
                if c > 0:
                    heapq.heappush(heap, (-c, pair))
            pair_count.pop(best, None)

            if (m + 1) % log_every == 0:
                print(f"  merge {m + 1:,}/{n_merges:,}  last = {vocab[-1]!r}")

        return cls(merges, vocab_size)

    # -------------------------------------------------------------- encoding
    def _encode_word(self, word):
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) > 1:
            best_rank, best_pair, best_new = None, None, None
            for pair in zip(ids, ids[1:]):
                hit = ranks.get(pair)
                if hit is not None and (best_rank is None or hit[0] < best_rank):
                    best_rank, best_pair, best_new = hit[0], pair, hit[1]
            if best_pair is None:
                break
            a, b = best_pair
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(best_new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out
        return ids

    def encode(self, text):
        cache = self._cache
        if len(cache) > 1_000_000:               # keep RAM bounded on huge corpora
            cache.clear()
        out = []
        for word in PRETOKEN_RE.findall(text):
            ids = cache.get(word)
            if ids is None:
                ids = cache[word] = self._encode_word(word)
            out.extend(ids)
        return out

    def decode(self, ids):
        specials = (self.eos_id, self.pad_id)
        data = b"".join(self.vocab[i] for i in ids if i < len(self.vocab) and i not in specials)
        return data.decode("utf-8", errors="replace")

    # ----------------------------------------------------------- persistence
    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges,
                       "eos": EOS_TOKEN, "pad": PAD_TOKEN}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            blob = json.load(f)
        return cls(blob["merges"], blob["vocab_size"])

## 6. Data prep: train the tokenizer, tokenize the corpus, mirror to the Hub
Uses every CPU core. Skipped automatically when the files are already there.

In [ ]:
%%writefile /kaggle/working/arch2/prep_data.py
"""
One-off data preparation (skipped automatically for anything already on disk):

  1. learn the byte-level BPE merges from a spread-out sample of the training text
  2. tokenize the corpus with all CPU cores into flat uint16 files (train.bin / val.bin)
  3. mirror tokenizer + bins to the Hugging Face repo so a Kaggle restart never repeats this

Training later memory-maps those files, so RAM usage during training stays tiny.
"""
import os
import sys
import json
import time
from collections import deque
from multiprocessing import Pool

import numpy as np
import ijson
from huggingface_hub import hf_hub_download, HfApi

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
from bpe_tokenizer import ByteBPE

TOKEN = os.environ.get("HF_TOKEN")


def local_train_files():
    paths = []
    for name in C.TRAIN_FILES:
        paths.append(hf_hub_download(C.HF_DATASET_REPO, name, repo_type="dataset", token=TOKEN))
    return paths


def iter_texts(path):
    """Stream the 'text' field of every record without loading the whole json file."""
    with open(path, "rb") as f:
        for rec in ijson.items(f, "item"):
            text = rec.get("text", "")
            if len(text) >= C.MIN_DOC_CHARS:
                yield text


# ------------------------------------------------------------------ stage 1
def tokenizer_sample(paths):
    budget = C.TOK_SAMPLE_MB_PER_FILE * 1_000_000
    for p in paths:
        used = 0
        for i, text in enumerate(iter_texts(p)):
            if i % C.TOK_SAMPLE_STRIDE:
                continue
            yield text
            used += len(text)
            if used >= budget:
                break
        print(f"  sampled {used / 1e6:.0f} MB from {os.path.basename(p)}", flush=True)


def train_tokenizer(paths):
    print("[1/2] Training BPE tokenizer ...", flush=True)
    t0 = time.time()
    tok = ByteBPE.train(tokenizer_sample(paths), vocab_size=C.VOCAB_SIZE)
    tok.save(C.TOKENIZER_PATH)
    print(f"  done in {(time.time() - t0) / 60:.1f} min, {len(tok.merges):,} merges", flush=True)


# ------------------------------------------------------------------ stage 2
_tok = None


def _init_worker(path):
    global _tok
    _tok = ByteBPE.load(path)


def _encode_batch(batch):
    """batch: list of (is_val, text). Every doc is stored as <eos> + tokens."""
    train, val = [], []
    for is_val, text in batch:
        ids = [_tok.eos_id]
        ids.extend(_tok.encode(text))
        (val if is_val else train).append(np.asarray(ids, dtype=np.uint16))
    cat = lambda xs: np.concatenate(xs) if xs else np.empty(0, dtype=np.uint16)
    return cat(train), cat(val)


def interleaved_texts(paths, weights):
    """
    Walk all source files at once and always pull the next document from the source that is
    furthest behind its weighted share (measured in characters). A source that ends is dropped.
    """
    streams = [iter_texts(p) for p in paths]
    taken = [0.0] * len(paths)
    alive = [True] * len(paths)
    while any(alive):
        i = min((j for j in range(len(paths)) if alive[j]), key=lambda j: taken[j] / weights[j])
        try:
            text = next(streams[i])
        except StopIteration:
            alive[i] = False
            print(f"  source exhausted: {os.path.basename(paths[i])}", flush=True)
            continue
        taken[i] += len(text)
        yield text


def doc_batches(paths, max_chars=2_000_000):
    batch, chars, n = [], 0, 0
    for text in interleaved_texts(paths, C.SOURCE_WEIGHTS):
        batch.append((n % C.VAL_EVERY_NTH_DOC == 0, text))
        chars += len(text)
        n += 1
        if chars >= max_chars:
            yield batch
            batch, chars = [], 0
    if batch:
        yield batch


def tokenize_corpus(paths):
    print(f"[2/2] Tokenizing with {C.PREP_WORKERS} workers -> {C.TRAIN_TOKENS / 1e6:.0f}M train tokens", flush=True)
    t0 = time.time()
    n_train = n_val = 0
    next_report = 20_000_000
    window = 3 * C.PREP_WORKERS               # keep only a few batches in flight -> bounded RAM

    with open(C.TRAIN_BIN, "wb") as ftrain, open(C.VAL_BIN, "wb") as fval, \
            Pool(C.PREP_WORKERS, initializer=_init_worker, initargs=(C.TOKENIZER_PATH,)) as pool:

        def absorb(result):
            nonlocal n_train, n_val, next_report
            tr, va = result.get()
            if n_val < C.VAL_TOKENS and len(va):
                va = va[: C.VAL_TOKENS - n_val]
                fval.write(va.tobytes())
                n_val += len(va)
            if n_train < C.TRAIN_TOKENS + 1 and len(tr):
                tr = tr[: C.TRAIN_TOKENS + 1 - n_train]     # +1 so the last block has a target
                ftrain.write(tr.tobytes())
                n_train += len(tr)
            if n_train >= next_report:
                rate = n_train / (time.time() - t0)
                eta = (C.TRAIN_TOKENS - n_train) / max(rate, 1) / 60
                print(f"  {n_train / 1e6:6.0f}M tokens | {rate / 1e3:.0f}K tok/s | eta {eta:.1f} min", flush=True)
                next_report += 20_000_000

        pending = deque()
        for batch in doc_batches(paths):
            pending.append(pool.apply_async(_encode_batch, (batch,)))
            if len(pending) >= window:
                absorb(pending.popleft())
            if n_train >= C.TRAIN_TOKENS + 1:
                break
        while pending and n_train < C.TRAIN_TOKENS + 1:
            absorb(pending.popleft())
        pool.terminate()

    if n_train < C.TRAIN_TOKENS:
        print(f"  WARNING: corpus only yielded {n_train:,} tokens; training will loop over it.", flush=True)
    meta = {"train_tokens": n_train, "val_tokens": n_val, "vocab_size": C.VOCAB_SIZE, "dtype": "uint16"}
    with open(C.META_PATH, "w") as f:
        json.dump(meta, f)
    print(f"  finished in {(time.time() - t0) / 60:.1f} min: {n_train:,} train / {n_val:,} val tokens", flush=True)


# ------------------------------------------------------------------- hub
def mirror_to_hub():
    api = HfApi()
    api.create_repo(C.HF_MODEL_REPO, token=TOKEN, private=C.HF_PRIVATE, exist_ok=True)
    # meta goes last: its presence on the hub means the whole upload finished
    for local in (C.TOKENIZER_PATH, C.VAL_BIN, C.TRAIN_BIN, C.META_PATH):
        name = "data/" + os.path.basename(local)
        print(f"  uploading {name} ...", flush=True)
        api.upload_file(path_or_fileobj=local, path_in_repo=name, repo_id=C.HF_MODEL_REPO, token=TOKEN)


def main():
    os.makedirs(C.DATA_DIR, exist_ok=True)
    have_bins = all(os.path.exists(p) for p in (C.TRAIN_BIN, C.VAL_BIN, C.META_PATH, C.TOKENIZER_PATH))
    if have_bins:
        print("Tokenizer and token files already present - nothing to do.")
        return
    paths = local_train_files()
    if not os.path.exists(C.TOKENIZER_PATH):
        train_tokenizer(paths)
    tokenize_corpus(paths)
    print("Mirroring to Hugging Face ...", flush=True)
    mirror_to_hub()
    print("Data prep complete.")


if __name__ == "__main__":
    main()

In [ ]:
import subprocess
rc = subprocess.run([sys.executable, "prep_data.py"], cwd=WORK_DIR).returncode
assert rc == 0, "data prep failed - read the log above"

## 7. Tokenizer sanity check

In [ ]:
import numpy as np
from bpe_tokenizer import ByteBPE

tok = ByteBPE.load(C.TOKENIZER_PATH)
sample = "The Hon'ble Supreme Court held that Section 438 of the Cr.P.C. does not apply; see (2013) 5 SCC 1."
ids = tok.encode(sample)
assert tok.decode(ids) == sample, "round-trip failed"
print(f"vocab {tok.vocab_size} | eos {tok.eos_id} | pad {tok.pad_id}")
print(f"{len(sample)} chars -> {len(ids)} tokens:", [tok.vocab[i].decode('utf-8', 'replace') for i in ids[:14]], "...")

meta = json.load(open(C.META_PATH))
train = np.memmap(C.TRAIN_BIN, dtype=np.uint16, mode="r")
print(f"train tokens: {meta['train_tokens']:,} | val tokens: {meta['val_tokens']:,} | max id {int(train[:5_000_000].max())}")
print("first tokens decoded:", repr(tok.decode(train[:60].tolist())))

## 8. Model
Bidirectional encoder + causal decoder with cross-attention. Grouped-query attention, RoPE and RMSNorm throughout.

In [ ]:
%%writefile /kaggle/working/arch2/model.py
"""
Arch 2: a compact encoder-decoder (T5-style) language model, written from scratch.

It is shaped like the task itself: the ENCODER reads the 512-word prompt with full bidirectional
attention, and the DECODER writes the continuation, attending to the prompt through cross-attention.

Same efficiency toolbox as the other architectures:
  * RMSNorm (pre-norm), no biases, SwiGLU feed-forward
  * RoPE inside self-attention (cross-attention needs none: the encoder output already carries position)
  * grouped-query attention everywhere (fewer KV heads than query heads)
  * everything runs through F.scaled_dot_product_attention (fused kernels, no T x T matrix)
  * one embedding table shared by encoder input, decoder input and the output head
  * KV-cache decoding with the cross-attention K/V computed once per prompt
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_head: int = 12
    n_kv_head: int = 4
    n_enc_layer: int = 6
    n_dec_layer: int = 6
    d_ff: int = 2048
    enc_len: int = 1024            # prompt tokens the encoder sees
    dec_len: int = 320             # continuation tokens the decoder is trained on
    rope_base: float = 10000.0
    norm_eps: float = 1e-5


class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32 * self.weight).to(x.dtype)


def build_rope_cache(head_dim, max_len, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


def repeat_kv(x, group):
    return x if group == 1 else x.repeat_interleave(group, dim=1)


class SelfAttention(nn.Module):
    """Grouped-query self-attention with RoPE. `causal` picks decoder (True) or encoder (False) behaviour."""

    def __init__(self, cfg, causal):
        super().__init__()
        self.causal = causal
        self.n_head, self.n_kv = cfg.n_head, cfg.n_kv_head
        self.head_dim = cfg.d_model // cfg.n_head
        self.group = cfg.n_head // cfg.n_kv_head
        self.q_dim = self.n_head * self.head_dim
        self.kv_dim = self.n_kv * self.head_dim
        self.qkv = nn.Linear(cfg.d_model, self.q_dim + 2 * self.kv_dim, bias=False)
        self.proj = nn.Linear(self.q_dim, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, past=None, key_mask=None):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split([self.q_dim, self.kv_dim, self.kv_dim], dim=-1)
        q = apply_rope(q.view(B, T, self.n_head, self.head_dim).transpose(1, 2), cos, sin)
        k = apply_rope(k.view(B, T, self.n_kv, self.head_dim).transpose(1, 2), cos, sin)
        v = v.view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        if past is not None:
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        present = (k, v)
        k, v = repeat_kv(k, self.group), repeat_kv(v, self.group)

        if key_mask is not None:                       # padded prompt (inference) or single-step decoding
            o = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)
        else:
            o = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and past is None and T > 1)
        return self.proj(o.transpose(1, 2).reshape(B, T, self.q_dim)), present


class CrossAttention(nn.Module):
    """Decoder queries attend to the encoder output. No positional rotation here."""

    def __init__(self, cfg):
        super().__init__()
        self.n_head, self.n_kv = cfg.n_head, cfg.n_kv_head
        self.head_dim = cfg.d_model // cfg.n_head
        self.group = cfg.n_head // cfg.n_kv_head
        self.q_dim = self.n_head * self.head_dim
        self.kv_dim = self.n_kv * self.head_dim
        self.q = nn.Linear(cfg.d_model, self.q_dim, bias=False)
        self.kv = nn.Linear(cfg.d_model, 2 * self.kv_dim, bias=False)
        self.proj = nn.Linear(self.q_dim, cfg.d_model, bias=False)

    def project_kv(self, enc_out):
        B, S, _ = enc_out.shape
        k, v = self.kv(enc_out).split([self.kv_dim, self.kv_dim], dim=-1)
        return (k.view(B, S, self.n_kv, self.head_dim).transpose(1, 2),
                v.view(B, S, self.n_kv, self.head_dim).transpose(1, 2))

    def forward(self, x, enc_out=None, cross_kv=None, enc_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k, v = cross_kv if cross_kv is not None else self.project_kv(enc_out)
        k, v = repeat_kv(k, self.group), repeat_kv(v, self.group)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=enc_mask)
        return self.proj(o.transpose(1, 2).reshape(B, T, self.q_dim))


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.gate_up = nn.Linear(d_model, 2 * hidden, bias=False)
        self.down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        gate, up = self.gate_up(x).chunk(2, dim=-1)
        return self.down(F.silu(gate) * up)


class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = SelfAttention(cfg, causal=False)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff)

    def forward(self, x, cos, sin, key_mask=None):
        a, _ = self.attn(self.attn_norm(x), cos, sin, key_mask=key_mask)
        x = x + a
        return x + self.ffn(self.ffn_norm(x))


class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.self_attn = SelfAttention(cfg, causal=True)
        self.cross_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.cross_attn = CrossAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff)

    def forward(self, x, cos, sin, enc_out=None, cross_kv=None, enc_mask=None, past=None, key_mask=None):
        a, present = self.self_attn(self.self_norm(x), cos, sin, past=past, key_mask=key_mask)
        x = x + a
        x = x + self.cross_attn(self.cross_norm(x), enc_out=enc_out, cross_kv=cross_kv, enc_mask=enc_mask)
        return x + self.ffn(self.ffn_norm(x)), present


class EncDecLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)          # shared by encoder, decoder and output head
        self.enc_layers = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_enc_layer)])
        self.enc_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.dec_layers = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_dec_layer)])
        self.dec_norm = RMSNorm(cfg.d_model, cfg.norm_eps)

        cos, sin = build_rope_cache(cfg.d_model // cfg.n_head, max(cfg.enc_len, 2 * cfg.dec_len), cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        n_res = cfg.n_enc_layer + 2 * cfg.n_dec_layer                      # residual writers: shrink them with depth
        scale = 0.02 / math.sqrt(n_res)
        for layer in self.enc_layers:
            nn.init.normal_(layer.attn.proj.weight, std=scale)
            nn.init.normal_(layer.ffn.down.weight, std=scale)
        for layer in self.dec_layers:
            nn.init.normal_(layer.self_attn.proj.weight, std=scale)
            nn.init.normal_(layer.cross_attn.proj.weight, std=scale)
            nn.init.normal_(layer.ffn.down.weight, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def active_params(self):                                              # dense model: everything is active
        return self.num_params()

    # ---- pieces ------------------------------------------------------------
    def encode(self, enc_ids, enc_mask=None):
        """enc_mask: optional (B, S) bool, True for real tokens (only needed for padded batches)."""
        B, S = enc_ids.shape
        cos, sin = self.rope_cos[:S], self.rope_sin[:S]
        key_mask = None if enc_mask is None else enc_mask[:, None, None, :]
        x = self.tok_emb(enc_ids)
        for layer in self.enc_layers:
            x = layer(x, cos, sin, key_mask)
        return self.enc_norm(x)

    def hidden_states(self, idx):
        """Encoder states (B, T, d_model): what a Phase 2-7 task head sits on top of."""
        return self.encode(idx)

    def cross_kv(self, enc_out):
        return [layer.cross_attn.project_kv(enc_out) for layer in self.dec_layers]

    def decode(self, dec_ids, enc_out=None, cross_kvs=None, enc_mask=None, caches=None, start_pos=0, last_only=False):
        B, T = dec_ids.shape
        cos, sin = self.rope_cos[start_pos:start_pos + T], self.rope_sin[start_pos:start_pos + T]
        cross_mask = None if enc_mask is None else enc_mask[:, None, None, :]
        x = self.tok_emb(dec_ids)
        new_caches = []
        for i, layer in enumerate(self.dec_layers):
            x, present = layer(x, cos, sin, enc_out=enc_out,
                               cross_kv=None if cross_kvs is None else cross_kvs[i], enc_mask=cross_mask,
                               past=None if caches is None else caches[i])
            new_caches.append(present)
        x = self.dec_norm(x)
        if last_only:
            x = x[:, -1:, :]
        return F.linear(x, self.tok_emb.weight), new_caches

    # ---- training forward --------------------------------------------------
    def forward(self, enc_ids, dec_ids, targets):
        """Teacher-forced: returns (cross-entropy over the decoder tokens, z-loss)."""
        enc_out = self.encode(enc_ids)
        logits, _ = self.decode(dec_ids, enc_out=enc_out)
        logits = logits.float()
        ce = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        z = torch.logsumexp(logits, dim=-1).pow(2).mean()
        return ce, z


# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
def _sample(logits, temperature, top_k, top_p):
    logits = logits / max(temperature, 1e-5)
    k = min(top_k, logits.size(-1)) if top_k > 0 else logits.size(-1)
    vals, idx = torch.topk(logits, k, dim=-1)
    probs = F.softmax(vals, dim=-1)
    if top_p < 1.0:
        before = probs.cumsum(-1) - probs
        vals = vals.masked_fill(before >= top_p, float("-inf"))
        probs = F.softmax(vals, dim=-1)
    return idx.gather(-1, torch.multinomial(probs, 1)).squeeze(-1)


@torch.no_grad()
def generate(model, prompts, tokenizer, max_new_tokens=320, target_words=128,
             temperature=0.7, top_k=50, top_p=0.92, check_every=8, suppress_eos=True):
    """
    prompts: list of token-id lists (<= enc_len each). The prompts are left-padded into one batch and the
    padding is masked out of both the encoder and the cross-attention. Returns one continuation string each.
    """
    model.eval()
    dev = next(model.parameters()).device
    B = len(prompts)
    S = max(len(p) for p in prompts)
    assert S <= model.cfg.enc_len, "prompt longer than the encoder context"
    assert max_new_tokens <= 2 * model.cfg.dec_len, "continuation longer than the cached RoPE table"
    pad, eos = tokenizer.pad_id, tokenizer.eos_id

    ids = torch.full((B, S), pad, dtype=torch.long)
    valid = torch.zeros((B, S), dtype=torch.bool)
    for r, p in enumerate(prompts):
        ids[r, S - len(p):] = torch.tensor(p, dtype=torch.long)
        valid[r, S - len(p):] = True
    ids, valid = ids.to(dev), valid.to(dev)

    enc_out = model.encode(ids, valid)
    cross_kvs = model.cross_kv(enc_out)

    cur = torch.full((B, 1), eos, dtype=torch.long, device=dev)            # decoder start token
    out = torch.empty((B, 0), dtype=torch.long, device=dev)
    finished = torch.zeros(B, dtype=torch.bool, device=dev)
    caches = None
    for step in range(max_new_tokens):
        logits, caches = model.decode(cur, cross_kvs=cross_kvs, enc_mask=valid, caches=caches,
                                      start_pos=step, last_only=True)
        step_logits = logits[:, -1].float()
        step_logits[:, pad] = float("-inf")
        if suppress_eos:
            step_logits[:, eos] = float("-inf")
        nxt = _sample(step_logits, temperature, top_k, top_p)
        nxt = torch.where(finished, torch.full_like(nxt, pad), nxt)
        out = torch.cat([out, nxt[:, None]], dim=1)
        finished |= nxt == eos
        cur = nxt[:, None]

        if (step + 1) % check_every == 0:
            for r in torch.nonzero(~finished).flatten().tolist():
                if len(tokenizer.decode(out[r].tolist()).split()) >= target_words:
                    finished[r] = True
        if bool(finished.all()):
            break

    texts = []
    for r in range(B):
        toks = out[r].tolist()
        if eos in toks:
            toks = toks[:toks.index(eos)]
        toks = [t for t in toks if t != pad]
        texts.append(" ".join(tokenizer.decode(toks).split()[:target_words]))
    return texts

## 9. Model checks (CPU, tiny)
Parameter count of the real config, then correctness checks: cached incremental decoding must reproduce the teacher-forced pass, and left-padding a prompt must not change what the model sees.

In [ ]:
import model as M
importlib.reload(M)

real = M.ModelConfig(vocab_size=C.VOCAB_SIZE, d_model=C.D_MODEL, n_head=C.N_HEAD, n_kv_head=C.N_KV_HEAD,
                     n_enc_layer=C.N_ENC_LAYER, n_dec_layer=C.N_DEC_LAYER, d_ff=C.D_FF,
                     enc_len=C.ENC_LEN, dec_len=C.DEC_LEN)
with torch.device("meta"):                      # count parameters without allocating them
    n_params = M.EncDecLM(real).num_params()
print(f"Real config: {n_params / 1e6:.1f}M parameters")
assert 50e6 <= n_params <= 300e6, "outside the 50-300M budget"

torch.manual_seed(0)
tiny = M.EncDecLM(M.ModelConfig(vocab_size=300, d_model=64, n_head=4, n_kv_head=2, n_enc_layer=2, n_dec_layer=2,
                                d_ff=96, enc_len=24, dec_len=12))

# 1) training forward/backward runs and every parameter receives a gradient
tiny.train()
enc, dec_in, tgt = torch.randint(0, 300, (2, 24)), torch.randint(0, 300, (2, 12)), torch.randint(0, 300, (2, 12))
ce, z = tiny(enc, dec_in, tgt)
(ce + 1e-4 * z).backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in tiny.parameters()), "missing/non-finite gradient"
print(f"train step OK | ce {ce.item():.3f}")

# 2) incremental decoding with both caches == teacher-forced decoding
tiny.eval()
enc_out = tiny.encode(enc)
full, _ = tiny.decode(dec_in, enc_out=enc_out)
cross, caches, steps = tiny.cross_kv(enc_out), None, []
for t in range(12):
    lg, caches = tiny.decode(dec_in[:, t:t + 1], cross_kvs=cross, caches=caches, start_pos=t)
    steps.append(lg)
diff = (torch.cat(steps, 1) - full).abs().max().item()
print(f"Cached vs teacher-forced decoding, max abs diff: {diff:.2e}")
assert diff < 1e-4
del tiny

## 10. Trainer (DDP, runs under torchrun)
One process per GPU. Batch order is a pure function of the step number, so resuming is exact and free.

In [ ]:
%%writefile /kaggle/working/arch2/train.py
"""
Distributed trainer for arch 2 (encoder-decoder). Launched with torchrun, one process per GPU:

    torchrun --standalone --nproc_per_node=2 train.py

Design notes
  * Data: memory-mapped uint16 token file. Every training example is a window of ENC_LEN + DEC_LEN
    tokens: the first part is the prompt for the encoder, the rest is what the decoder must write.
    Window order is a seeded permutation, so the batch for step N is a pure function of N and
    resuming costs nothing.
  * Time budget: the LR follows max(step progress, wall-clock progress), so the cosine decay ends
    exactly when TRAIN_TIME_BUDGET_MIN runs out. Rank 0 owns the clock and broadcasts it.
  * Every CKPT_EVERY steps rank 0 saves the full training state and pushes it to Hugging Face from
    a background thread, so a Kaggle crash/timeout loses at most a few minutes.
"""
import os
import sys
import json
import math
import time
import queue
import threading

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
from model import EncDecLM, ModelConfig

# If Inductor ever fails to compile some part of the graph, run that part eagerly instead of crashing the whole job.
torch._dynamo.config.suppress_errors = True

T_START = time.time()


# ------------------------------------------------------------------ dist setup
rank = int(os.environ.get("RANK", 0))
world = int(os.environ.get("WORLD_SIZE", 1))
local_rank = int(os.environ.get("LOCAL_RANK", 0))
use_cuda = torch.cuda.is_available()
distributed = world > 1

if distributed:
    dist.init_process_group("nccl" if use_cuda else "gloo")
if use_cuda:
    torch.cuda.set_device(local_rank)
device = torch.device("cuda", local_rank) if use_cuda else torch.device("cpu")
is_main = rank == 0

torch.manual_seed(C.SEED)
if use_cuda:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")


def say(*a):
    if is_main:
        print(*a, flush=True)


# ------------------------------------------------------------------------ data
class TokenFile:
    """Random-access windows of EXAMPLE_LEN tokens from a flat uint16 token file."""

    def __init__(self, path, block, seed=None):
        self.data = np.memmap(path, dtype=np.uint16, mode="r")
        self.block = block
        self.n_blocks = len(self.data) // block
        self.order = np.random.default_rng(seed).permutation(self.n_blocks) if seed is not None else None

    def get(self, seq_ids):
        B = self.block
        rows = []
        for s in seq_ids:
            b = int(s) % self.n_blocks
            if self.order is not None:
                b = int(self.order[b])
            rows.append(self.data[b * B: b * B + B])
        return torch.from_numpy(np.stack(rows).astype(np.int64))


def split_example(window):
    """(B, ENC_LEN + DEC_LEN) -> encoder ids, decoder inputs (shifted right, starts with <eos>), decoder targets."""
    enc = window[:, :C.ENC_LEN]
    target = window[:, C.ENC_LEN:]
    start = torch.full_like(target[:, :1], EOS_ID)
    dec_in = torch.cat([start, target[:, :-1]], dim=1)
    return enc, dec_in, target


class Prefetcher:
    """Background thread that keeps a few ready-to-go micro-batches in a queue."""

    def __init__(self, stream, micro, step_stride, start_step, total_steps, accum):
        self.q = queue.Queue(maxsize=6)
        self.args = (stream, micro, step_stride, start_step, total_steps, accum)
        threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        stream, micro, stride, start, total, accum = self.args
        for step in range(start, total):
            for j in range(accum):
                base = step * stride + (j * world + rank) * micro
                w = stream.get(range(base, base + micro))
                self.q.put(w.pin_memory() if use_cuda else w)
        self.q.put(None)

    def next(self):
        return self.q.get().to(device, non_blocking=True)


# --------------------------------------------------------------------- hub sync
class HubSync:
    """Uploads files in a background thread; never blocks the training loop for long."""

    def __init__(self):
        self.thread = None
        self.pushes = 0
        self.api = None
        if is_main and os.environ.get("HF_TOKEN"):
            from huggingface_hub import HfApi
            self.api = HfApi()
            self.api.create_repo(C.HF_MODEL_REPO, token=os.environ["HF_TOKEN"], private=C.HF_PRIVATE, exist_ok=True)

    def wait(self):
        if self.thread is not None:
            self.thread.join()
            self.thread = None

    def push(self, files, message, block=False):
        if self.api is None:
            return
        self.wait()                        # the previous upload must be done before files get rewritten

        def work():
            try:
                for local, remote in files:
                    self.api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=C.HF_MODEL_REPO,
                                         token=os.environ["HF_TOKEN"], commit_message=message)
                self.pushes += 1
                print(f"  -> pushed to HF: {message}", flush=True)
                if self.pushes % C.SQUASH_EVERY_N_PUSHES == 0:
                    self.api.super_squash_history(repo_id=C.HF_MODEL_REPO, token=os.environ["HF_TOKEN"])
            except Exception as e:                                   # a failed push must not kill training
                print(f"  -> HF push failed ({type(e).__name__}: {e}); will retry at the next checkpoint", flush=True)

        self.thread = threading.Thread(target=work)
        self.thread.start()
        if block:
            self.wait()


# ----------------------------------------------------------------------- helpers
def lr_at(step, progress):
    warm = min(1.0, (step + 1) / C.WARMUP_STEPS)
    floor = C.PEAK_LR * C.MIN_LR_RATIO
    cosine = floor + 0.5 * (C.PEAK_LR - floor) * (1 + math.cos(math.pi * min(1.0, progress)))
    return cosine * warm


def make_optimizer(model):
    decay = [p for p in model.parameters() if p.ndim >= 2]
    no_decay = [p for p in model.parameters() if p.ndim < 2]
    groups = [{"params": decay, "weight_decay": C.WEIGHT_DECAY},
              {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=C.PEAK_LR, betas=tuple(C.BETAS), eps=1e-8, fused=use_cuda)


def save_training_state(raw_model, optimizer, scaler, step, tokens_seen, best_val, train_seconds):
    os.makedirs(C.CKPT_DIR, exist_ok=True)
    tmp = C.STATE_PATH + ".tmp"
    torch.save({
        "model": raw_model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "step": step,
        "tokens_seen": tokens_seen,
        "best_val": best_val,
        "train_seconds": train_seconds,
        "model_cfg": raw_model.cfg.__dict__,
    }, tmp)
    os.replace(tmp, C.STATE_PATH)           # atomic: a crash never leaves a half-written checkpoint


@torch.no_grad()
def evaluate(model, val_stream, micro):
    model.eval()
    per_rank = max(1, C.EVAL_SEQS // (micro * world))
    total = torch.zeros(2, device=device)
    for i in range(per_rank):
        base = (i * world + rank) * micro
        enc, dec_in, target = split_example(val_stream.get(range(base, base + micro)).to(device))
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_cuda):
            ce, _ = model(enc, dec_in, target)
        total[0] += ce.detach()
        total[1] += 1
    if distributed:
        dist.all_reduce(total)
    model.train()
    return (total[0] / total[1]).item()


# ------------------------------------------------------------------------- main
def main():
    global EOS_ID
    with open(C.META_PATH) as f:
        meta = json.load(f)
    EOS_ID = C.VOCAB_SIZE - 2                       # tokenizer layout: ... <eos>, <pad> at the end of the vocab

    accum = C.EXAMPLES_PER_STEP // (C.MICRO_BATCH * world)
    assert accum >= 1 and accum * C.MICRO_BATCH * world == C.EXAMPLES_PER_STEP, \
        f"EXAMPLES_PER_STEP must split evenly over {world} GPUs x MICRO_BATCH {C.MICRO_BATCH}"
    tokens_per_step = C.EXAMPLES_PER_STEP * C.EXAMPLE_LEN
    total_steps = C.TRAIN_TOKENS // tokens_per_step
    budget_s = C.TRAIN_TIME_BUDGET_MIN * 60

    cfg = ModelConfig(vocab_size=C.VOCAB_SIZE, d_model=C.D_MODEL, n_head=C.N_HEAD, n_kv_head=C.N_KV_HEAD,
                      n_enc_layer=C.N_ENC_LAYER, n_dec_layer=C.N_DEC_LAYER, d_ff=C.D_FF,
                      enc_len=C.ENC_LEN, dec_len=C.DEC_LEN, rope_base=C.ROPE_BASE, norm_eps=C.NORM_EPS)
    raw_model = EncDecLM(cfg).to(device)
    n_params = raw_model.num_params()
    assert not C.ENFORCE_PARAM_BUDGET or 50e6 <= n_params <= 300e6, f"{n_params / 1e6:.1f}M params is outside the 50-300M budget"

    optimizer = make_optimizer(raw_model)
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)

    step, tokens_seen, best_val, used_s = 0, 0, float("inf"), 0.0
    if os.path.exists(C.STATE_PATH):
        ckpt = torch.load(C.STATE_PATH, map_location="cpu")
        raw_model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scaler.load_state_dict(ckpt["scaler"])
        step, tokens_seen, best_val = ckpt["step"], ckpt["tokens_seen"], ckpt["best_val"]
        used_s = ckpt.get("train_seconds", 0.0)
        del ckpt
        say(f"Resumed from checkpoint: step {step} / {total_steps} ({tokens_seen / 1e6:.0f}M tokens, "
            f"{used_s / 60:.0f} of {C.TRAIN_TIME_BUDGET_MIN} budget minutes already used)")
    else:
        say("No checkpoint found - starting from scratch.")

    say(f"GPUs: {world} | params: {n_params / 1e6:.1f}M | examples/step: {C.EXAMPLES_PER_STEP} "
        f"({tokens_per_step:,} corpus tokens) | micro-batch {C.MICRO_BATCH} x accum {accum} per GPU "
        f"| step cap: {total_steps} | time budget: {C.TRAIN_TIME_BUDGET_MIN} min")
    if meta["train_tokens"] < C.TRAIN_TOKENS:
        say(f"Note: only {meta['train_tokens']:,} tokens on disk, the data will be revisited (multiple epochs).")

    model = torch.compile(raw_model) if (C.COMPILE and use_cuda) else raw_model
    if distributed:
        model = DDP(model, device_ids=[local_rank] if use_cuda else None)
    model.train()

    train_stream = TokenFile(C.TRAIN_BIN, C.EXAMPLE_LEN, seed=C.SEED)
    val_stream = TokenFile(C.VAL_BIN, C.EXAMPLE_LEN)
    feeder = Prefetcher(train_stream, C.MICRO_BATCH, C.EXAMPLES_PER_STEP, step, total_steps, accum)
    hub = HubSync()

    def sync_ctx(is_last_micro):
        # skip the gradient all-reduce on all but the last micro-batch of a step
        if distributed and not is_last_micro:
            return model.no_sync()
        return _null()

    def progress_now():
        """Fraction of the run that is over: whichever is further along, steps or wall-clock. Same on every rank."""
        p = torch.tensor(max(step / total_steps, (used_s + time.time() - T_START) / budget_s), device=device)
        if distributed:
            dist.broadcast(p, src=0)
        return float(p.item())

    log_file = open(C.LOG_PATH, "a") if is_main else None
    t_log = time.time()
    tok_since_log = 0
    loss_acc = torch.zeros((), device=device)
    steps_in_acc = 0
    grad_norm = torch.zeros((), device=device)
    crash_guard_s = C.TIME_LIMIT_HOURS * 3600
    interrupted = False
    train_seconds = lambda: used_s + time.time() - T_START

    progress = progress_now()
    while progress < 1.0 and step < total_steps:
        lr = lr_at(step, progress)
        for g in optimizer.param_groups:
            g["lr"] = lr

        for j in range(accum):
            enc, dec_in, target = split_example(feeder.next())
            with sync_ctx(j == accum - 1):
                with torch.autocast("cuda", dtype=torch.float16, enabled=use_cuda):
                    ce, z = model(enc, dec_in, target)
                    loss = (ce + C.Z_LOSS * z) / accum
                scaler.scale(loss).backward()
            loss_acc += ce.detach() / accum

        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), C.MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        step += 1
        steps_in_acc += 1
        tokens_seen += tokens_per_step
        tok_since_log += tokens_per_step
        progress = progress_now()

        if step % C.LOG_EVERY == 0:
            avg = (loss_acc / steps_in_acc).item()
            loss_acc.zero_()
            steps_in_acc = 0
            speed = tok_since_log / (time.time() - t_log)
            mem = torch.cuda.max_memory_allocated() / 1e9 if use_cuda else 0.0
            left_min = max(0.0, (budget_s - train_seconds()) / 60)
            say(f"step {step:5d} | loss {avg:.4f} | lr {lr:.2e} | gnorm {float(grad_norm):.2f} | {speed / 1e3:.1f}K tok/s "
                f"| mem {mem:.1f}GB | progress {100 * progress:.0f}% | {left_min:.0f} min left")
            if is_main:
                log_file.write(json.dumps({"step": step, "loss": avg, "lr": lr, "tok_s": speed}) + "\n")
                log_file.flush()
            t_log, tok_since_log = time.time(), 0

        if step % C.EVAL_EVERY == 0:
            val = evaluate(model, val_stream, C.MICRO_BATCH)
            improved = val < best_val
            best_val = min(best_val, val)
            say(f"  [eval] step {step} | val loss {val:.4f} | val ppl {math.exp(min(val, 20)):.2f}")
            if is_main:
                log_file.write(json.dumps({"step": step, "val_loss": val}) + "\n")
                log_file.flush()
                if improved:                          # keep the best-validation weights, not just the latest
                    torch.save({"model": {k: v.half() for k, v in raw_model.state_dict().items()},
                                "model_cfg": cfg.__dict__, "step": step, "val_loss": val}, C.BEST_PATH)
                    hub.push([(C.BEST_PATH, "model_best.pt")], f"best val {val:.4f} at step {step}")

        # crash guard against Kaggle's own wall: every rank must decide the same, so rank 0 broadcasts
        if step % 10 == 0:
            flag = torch.tensor(1.0 if (time.time() - T_START) > crash_guard_s else 0.0, device=device)
            if distributed:
                dist.broadcast(flag, src=0)
            interrupted = bool(flag.item())

        if step % C.CKPT_EVERY == 0 or interrupted:
            if is_main:
                hub.wait()
                save_training_state(raw_model, optimizer, scaler, step, tokens_seen, best_val, train_seconds())
                hub.push([(C.STATE_PATH, "training_state.pt"), (C.LOG_PATH, "train_log.jsonl")],
                         f"checkpoint step {step}", block=interrupted)
            if distributed:
                dist.barrier()
        if interrupted:
            say(f"Crash guard reached at step {step}. State saved and pushed - re-run the training cell to continue.")
            break

    if not interrupted:
        # finished: final validation, final state, final weights
        val = evaluate(model, val_stream, C.MICRO_BATCH)
        improved = val < best_val
        best_val = min(best_val, val)
        say(f"  [final eval] step {step} | val loss {val:.4f} | val ppl {math.exp(min(val, 20)):.2f}")
        if is_main:
            log_file.write(json.dumps({"step": step, "val_loss": val}) + "\n")
            log_file.flush()
            if improved:
                torch.save({"model": {k: v.half() for k, v in raw_model.state_dict().items()},
                            "model_cfg": cfg.__dict__, "step": step, "val_loss": val}, C.BEST_PATH)
            hub.wait()
            save_training_state(raw_model, optimizer, scaler, step, tokens_seen, best_val, train_seconds())
            torch.save({"model": {k: v.half() for k, v in raw_model.state_dict().items()},
                        "model_cfg": cfg.__dict__}, C.FINAL_PATH)
            files = [(C.FINAL_PATH, "model_final.pt"), (C.TOKENIZER_PATH, "tokenizer.json"),
                     (C.STATE_PATH, "training_state.pt"), (C.LOG_PATH, "train_log.jsonl")]
            if os.path.exists(C.BEST_PATH):
                files.append((C.BEST_PATH, "model_best.pt"))
            hub.push(files, "final model", block=True)
            say(f"Training complete in {train_seconds() / 60:.0f} min ({step} steps, {tokens_seen / 1e6:.0f}M tokens). Final weights pushed.")
    hub.wait()
    if distributed:
        dist.destroy_process_group()


class _null:
    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False


if __name__ == "__main__":
    main()

## 11. Train
Launches `torchrun` on every visible GPU and streams its log here.

* Re-running this cell after a crash resumes from the last checkpoint on the Hub.
* The first ~2-4 minutes are `torch.compile` warming up - the first log line appears after that. Compile time counts against the budget.
* Lines look like `step 400 | loss ... | lr ... | K tok/s | mem ...GB | progress 31% | 55 min left`. `progress` is whichever is further along, steps or wall-clock; the run ends at 100% with the LR fully decayed. `[eval]` lines show validation loss.
* The run finishes by itself after `TRAIN_TIME_BUDGET_MIN` minutes, runs a final validation, and pushes the final weights.

In [ ]:
import subprocess

n_gpu = max(1, torch.cuda.device_count())
env = dict(os.environ)
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["OMP_NUM_THREADS"] = "4"
if C.NCCL_P2P_OFF:
    env["NCCL_P2P_DISABLE"] = "1"

cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={n_gpu}", "train.py"]
print("Launching:", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=WORK_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
print(f"\ntrainer exited with code {rc}")

## 12. Loss curves

In [ ]:
import matplotlib.pyplot as plt

train_pts, val_pts = [], []
with open(C.LOG_PATH) as f:
    for line in f:
        r = json.loads(line)
        if "loss" in r:
            train_pts.append((r["step"], r["loss"]))
        elif "val_loss" in r:
            val_pts.append((r["step"], r["val_loss"]))

plt.figure(figsize=(11, 4.5))
plt.plot(*zip(*train_pts), label="train", alpha=0.8)
if val_pts:
    plt.plot(*zip(*val_pts), "o-", label="validation", color="orange")
plt.xlabel("optimizer step"); plt.ylabel("cross-entropy"); plt.yscale("log")
plt.title("Arch 2 encoder-decoder - training and validation loss")
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
plot_path = os.path.join(C.WORK_DIR, "loss_curve.png")
plt.savefig(plot_path, dpi=150); plt.show()
if val_pts:
    print(f"final val loss {val_pts[-1][1]:.4f} | perplexity {math.exp(val_pts[-1][1]):.2f}")

## 13. Load the trained model for generation

In [ ]:
# Which weights to generate with: "final" (end of training), "best" (lowest validation loss) or "latest" (last saved state).
# Section 13b compares them on held-out text, so you can pick by generation quality, not only validation loss.
CHOSEN_WEIGHTS = "final"

on_hub_now = set(api.list_repo_files(C.HF_MODEL_REPO, token=HF_TOKEN))
WEIGHT_FILES = {"final": (C.FINAL_PATH, "model_final.pt"),
                "best": (C.BEST_PATH, "model_best.pt"),
                "latest": (C.STATE_PATH, "training_state.pt")}
available = {}
for name, (path, remote) in WEIGHT_FILES.items():
    if not os.path.exists(path) and remote in on_hub_now and name != "latest":
        hf_hub_download(C.HF_MODEL_REPO, remote, local_dir=C.CKPT_DIR, token=HF_TOKEN)
    if os.path.exists(path):
        available[name] = path
if "final" in available:
    available.pop("latest", None)                 # identical to final once training has finished
print("Weights found:", list(available))
if CHOSEN_WEIGHTS not in available:
    CHOSEN_WEIGHTS = next(k for k in ("final", "best", "latest") if k in available)
    print("Requested weights not available, using:", CHOSEN_WEIGHTS)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def load_model(path):
    blob = torch.load(path, map_location="cpu")
    model = M.EncDecLM(M.ModelConfig(**blob["model_cfg"]))
    model.load_state_dict({k: v.float() for k, v in blob["model"].items()})
    del blob
    return model.to(device).eval()

gen_model = load_model(available[CHOSEN_WEIGHTS])
gc.collect()
print(f"Generating with '{CHOSEN_WEIGHTS}' weights on {device}: {gen_model.num_params() / 1e6:.1f}M parameters")

## 13b. Local evaluation on held-out text (mirrors the official metrics)
The competition scores a 128-word continuation of a 512-word prompt with BERTScore F1 (primary), ROUGE-L F1 and BLEU-4.
Here we build the *same kind of task* from the validation documents (never trained on): first 512 words = prompt,
next 128 words = reference. ROUGE-L and BLEU-4 are implemented below; BERTScore is optional (it downloads roberta-large).
Use this to compare checkpoints by generation quality, not just validation loss.

In [ ]:
import numpy as np

def lcs_len(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]

def rouge_l_f1(ref, hyp):
    r, h = ref.split(), hyp.split()
    if not r or not h:
        return 0.0
    lcs = lcs_len(r, h)
    if lcs == 0:
        return 0.0
    p, rec = lcs / len(h), lcs / len(r)
    return 2 * p * rec / (p + rec)

def bleu4(ref, hyp):
    r, h = ref.split(), hyp.split()
    if not h:
        return 0.0
    log_p = 0.0
    for n in range(1, 5):
        ref_grams, hyp_grams = {}, {}
        for i in range(len(r) - n + 1):
            g = tuple(r[i:i + n]); ref_grams[g] = ref_grams.get(g, 0) + 1
        for i in range(len(h) - n + 1):
            g = tuple(h[i:i + n]); hyp_grams[g] = hyp_grams.get(g, 0) + 1
        overlap = sum(min(c, ref_grams.get(g, 0)) for g, c in hyp_grams.items())
        total = max(1, sum(hyp_grams.values()))
        log_p += math.log((overlap + 1) / (total + 1)) / 4          # add-one smoothing
    brevity = 1.0 if len(h) >= len(r) else math.exp(1 - len(r) / len(h))
    return brevity * math.exp(log_p)

# rebuild a few full documents from the validation token file (documents are separated by <eos>)
val_tokens = np.fromfile(C.VAL_BIN, dtype=np.uint16)
cuts = np.flatnonzero(val_tokens == tok.eos_id)
eval_prompts, eval_refs = [], []
for a, b in zip(cuts[:-1], cuts[1:]):                              # skip the last, possibly truncated, document
    words = tok.decode(val_tokens[a + 1:b].tolist()).split()
    if len(words) >= 512 + C.TARGET_WORDS:
        eval_prompts.append(" ".join(words[:512]))
        eval_refs.append(" ".join(words[512:512 + C.TARGET_WORDS]))
    if len(eval_prompts) >= C.N_LOCAL_EVAL:
        break
print(f"{len(eval_prompts)} held-out documents with 512-word prompts")

def run_generation(model, prompt_texts):
    room = gen_model.cfg.enc_len                      # the loaded checkpoint decides the prompt window
    ids = [tok.encode(t) for t in prompt_texts]
    ids = [x[-room:] if len(x) > room else [tok.eos_id] + x for x in ids]
    outs = []
    for b in range(0, len(ids), C.GEN_BATCH):
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            outs += M.generate(model, ids[b:b + C.GEN_BATCH], tok, max_new_tokens=C.MAX_NEW_TOKENS,
                               target_words=C.TARGET_WORDS, temperature=C.GEN_TEMPERATURE,
                               top_k=C.GEN_TOP_K, top_p=C.GEN_TOP_P)
    return outs

if eval_prompts:
    results = {}
    for name, path in available.items():
        mdl = gen_model if name == CHOSEN_WEIGHTS else load_model(path)
        hyps = run_generation(mdl, eval_prompts)
        rl = [rouge_l_f1(r, h) for r, h in zip(eval_refs, hyps)]
        bl = [bleu4(r, h) for r, h in zip(eval_refs, hyps)]
        results[name] = (float(np.mean(rl)), float(np.mean(bl)), hyps)
        print(f"{name:7s} | ROUGE-L F1 {results[name][0]:.4f} | BLEU-4 {results[name][1]:.4f} | "
              f"words/prediction {np.mean([len(h.split()) for h in hyps]):.0f}")
        if mdl is not gen_model:
            del mdl
            gc.collect()
            torch.cuda.empty_cache()
    hyps = results[CHOSEN_WEIGHTS][2]              # the rest of this section looks at the chosen weights

    RUN_BERTSCORE = False        # set True to also compute the official primary metric (needs: pip install bert_score)
    if RUN_BERTSCORE:
        from bert_score import score as bert_score
        _, _, f1 = bert_score(hyps, eval_refs, lang="en", model_type="roberta-large")
        print(f"BERTScore F1: {f1.mean().item():.4f}")
    print("\nSample prompt tail :", eval_prompts[0].split()[-25:])
    print("Sample reference   :", eval_refs[0][:200])
    print("Sample generation  :", hyps[0][:200])

## 14. Generate predictions for the test set
Encoder runs once per prompt; the decoder samples with a KV cache (temperature / top-k / top-p from `config.py`). Prompts are sorted by length so padding stays small.

In [ ]:
test_path = hf_hub_download(C.HF_DATASET_REPO, C.TEST_FILE, repo_type="dataset", token=HF_TOKEN)
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print("Test examples:", len(test_data))

room = gen_model.cfg.enc_len                      # the loaded checkpoint decides the prompt window
def make_prompt(text):
    ids = tok.encode(text)
    return ids[-room:] if len(ids) > room else [tok.eos_id] + ids

keys = list(test_data.keys())
n_trunc = sum(len(tok.encode(test_data[k]["text"])) > room for k in keys)
print(f"{n_trunc}/{len(keys)} prompts exceed {room} tokens and will be cut from the left (the doc says prompts are 512 words)")
prompts = {k: make_prompt(test_data[k]["text"]) for k in keys}
order = sorted(keys, key=lambda k: len(prompts[k]))

PRED_PATH = os.path.join(C.WORK_DIR, "prediction.json")
predictions = {}
t0 = time.time()
for b in range(0, len(order), C.GEN_BATCH):
    chunk = order[b:b + C.GEN_BATCH]
    with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
        outs = M.generate(gen_model, [prompts[k] for k in chunk], tok,
                          max_new_tokens=C.MAX_NEW_TOKENS, target_words=C.TARGET_WORDS,
                          temperature=C.GEN_TEMPERATURE, top_k=C.GEN_TOP_K, top_p=C.GEN_TOP_P)
    for k, text in zip(chunk, outs):
        predictions[str(k)] = {"text": text}
    done = len(predictions)
    if (b // C.GEN_BATCH) % 5 == 0 or done == len(order):
        print(f"  {done}/{len(order)} | {time.time() - t0:.0f}s elapsed")
    if (b // C.GEN_BATCH) % 20 == 0:                       # partial save in case the session dies
        with open(PRED_PATH, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False)

predictions = {str(k): predictions[str(k)] for k in keys}  # original order
print("Generation complete:", len(predictions), "predictions")

## 15. Save predictions and build the submission zip

In [ ]:
with open(PRED_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

SUBMISSION_ZIP = os.path.join(C.WORK_DIR, "submission.zip")
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_PATH, arcname="prediction.json")
print(f"Wrote {SUBMISSION_ZIP} ({os.path.getsize(SUBMISSION_ZIP) / 1024:.1f} KB)")

# --- submission self-check (the checklist in the competition document, Section 19)
official_ids = set(map(str, test_data.keys()))
assert set(predictions) == official_ids,     f"ID mismatch: missing {len(official_ids - set(predictions))}, unknown {len(set(predictions) - official_ids)}"
assert all(isinstance(v.get("text"), str) and v["text"].strip() for v in predictions.values()), "empty prediction text"
n_words = [len(v["text"].split()) for v in predictions.values()]
print(f"ID check passed ({len(predictions)} ids). Words per prediction: min {min(n_words)} / mean {np.mean(n_words):.0f} / max {max(n_words)}")
if min(n_words) < C.TARGET_WORDS:
    print(f"WARNING: {sum(w < C.TARGET_WORDS for w in n_words)} predictions are shorter than {C.TARGET_WORDS} words")
with zipfile.ZipFile(SUBMISSION_ZIP) as z:
    assert z.namelist() == ["prediction.json"], f"zip must contain only prediction.json at its root, got {z.namelist()}"
print("Zip structure OK: prediction.json at the archive root.")

for k in list(predictions)[:3]:
    words = predictions[k]["text"].split()
    print(f"\nExample {k} ({len(words)} words):\n{predictions[k]['text'][:300]}...")

# keep a copy on the hub too
api.upload_file(path_or_fileobj=SUBMISSION_ZIP, path_in_repo="submission.zip", repo_id=C.HF_MODEL_REPO, token=HF_TOKEN)
api.upload_file(path_or_fileobj=plot_path, path_in_repo="loss_curve.png", repo_id=C.HF_MODEL_REPO, token=HF_TOKEN)
print("Submission and loss curve pushed to", C.HF_MODEL_REPO)